# Decision curve analysis

5-year net benefit of SOC versus SOC+NMR from the fold-specific absolute risks in
`outputs/main/absolute_risks/` (`pred_risk_5yr`).
Risks are not recomputed from linear predictors. COPD is plotted separately; total CVD holds its slot in the primary grid.


In [ ]:
library(tidyverse)
library(patchwork)
library(ggpubr)
library(here)

source(here("helpers/oof_dca.r"))

fig_dir <- here("outputs/figures")
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

# set color palette and threshold ranges for DCA plots 
model_palette <- c("SOC" = "#E64B35", "SOC+NMR" = "#4DBBD5")
threshold_range <- seq(0.001, 0.50, by = 0.001)
t_star <- 5


In [ ]:
# load oof absolute risks 
risk_dir <- here("outputs/main/absolute_risks")

# set disease specifications 
disease_specs <- list(
  t2d = list(
    title = "T2D", prefix = "t2d_a1c_", primary = TRUE
  ),
  ckd = list(
    title = "CKD", prefix = "ckd_egfr_", primary = TRUE
  ),
  masld = list(
    title = "MASLD", prefix = "nafld_fib4_", primary = TRUE
  ),
  hf_cm = list(
    title = "Heart Failure", prefix = "hf_cm_prevent_ingredients_", primary = TRUE
  ),
  acute_mi = list(
    title = "Acute MI", prefix = "acute_mi_prevent_ingredients_", primary = TRUE
  ),
  chd = list(
    title = "CHD", prefix = "chd_revasc_prevent_ingredients_", primary = TRUE
  ),
  tia_stroke = list(
    title = "TIA/Stroke", prefix = "tia_stroke_prevent_ingredients_", primary = TRUE
  ),
  total_cvd = list(
    title = "Total CVD", prefix = "total_cvd_prevent_ingredients_", primary = TRUE
  ),
  afib = list(
    title = "Afib", prefix = "afib_chargeaf_ingredients_", primary = TRUE
  ),
  copd = list(
    title = "COPD", prefix = "copd_smoking_ldl_", primary = FALSE
  )
)

# set primary disease ids 
primary_ids <- names(Filter(function(spec) spec$primary, disease_specs))

# load oof absolute risks (SOC and SOC+NMR)
load_pair <- function(prefix) {
  list(
    SOC = readRDS(file.path(risk_dir, paste0(prefix, "marker_fold_baseline_risk.rds"))),
    `SOC+NMR` = readRDS(file.path(risk_dir, paste0(prefix, "marker_nmr_fold_baseline_risk.rds")))
  )
}

oof_store <- lapply(disease_specs, function(spec) load_pair(spec$prefix))

dca_curves <- lapply(oof_store, function(models) {
  lapply(models, function(obj) {
    compute_dca_df_from_oof_risk(
      obj,
      t_star = t_star,
      risk_col = "pred_risk_5yr",
      threshold_range = threshold_range
    )
  })
})


## Primary net-benefit curves

Nine diseases at 5 years. Y limits are set from each panel's own curves.


In [ ]:

p_dca_5yr <- {
  plots <- lapply(primary_ids, function(id) {
    plot_dca_compare(
      dca_curves[[id]],
      title = paste(disease_specs[[id]]$title, "Net Benefit"),
      x_limits = c(0.001, 0.50),
      palette = model_palette,
      t_star = t_star
    )
  })
  patchwork::wrap_plots(plots, ncol = 3, guides = "collect") +
    patchwork::plot_annotation(
      theme = theme(plot.title = element_text(face = "bold", hjust = 0.5))
    ) &
    theme(legend.position = "bottom")
}

options(repr.plot.height = 10, repr.plot.width = 10)
p_dca_5yr

ggsave(
  file.path(fig_dir, "DCA_5yr.png"),
  p_dca_5yr, width = 10, height = 10, dpi = 600
)


## Type 2 diabetes at a 20% threshold

The bar heights are the 5-year net benefit at threshold 0.20, rounded to one decimal.


In [ ]:
options(repr.plot.height = 4, repr.plot.width = 4)

plot_dca_compare(
  dca_curves$t2d,
  title = "T2D Net Benefit",
  x_limits = c(0.001, 0.50),
  palette = model_palette,
  t_star = 5
)


In [ ]:
options(repr.plot.height = 4, repr.plot.width = 4)
plot_dca_compare(
  dca_curves$t2d,
  title = "T2D Net Benefit",
  x_limits = c(0.001, 0.50),
  palette = model_palette,
  t_star = 5
)

t2d_nb <- vapply(
  dca_curves$t2d,
  nb_at_threshold,
  numeric(1),
  threshold = 0.20
)
t2d_nb

t2d_bar <- data.frame(
  model = factor(names(t2d_nb), levels = c("SOC", "SOC+NMR")),
  nb = round(unname(t2d_nb), 1)
)
delta_nb <- diff(t2d_bar$nb)
y_bracket <- max(t2d_bar$nb) + 1

options(repr.plot.height = 2, repr.plot.width = 5)
p_t2d_bar <- ggbarplot(
  t2d_bar, x = "model", y = "nb",
  fill = "model",
  palette = model_palette
) +
  geom_text(
    aes(label = sprintf("%.1f", nb)),
    hjust = 1.25, size = 3.5, color = "white", fontface = "bold"
  ) +
  annotate("segment", x = 1, xend = 2, y = y_bracket, yend = y_bracket) +
  annotate("segment", x = 1, xend = 1, y = y_bracket, yend = y_bracket - 0.4) +
  annotate("segment", x = 2, xend = 2, y = y_bracket, yend = y_bracket - 0.4) +
  annotate(
    "text", x = 1.5, y = y_bracket + 1.3,
    label = sprintf("+%.1f", delta_nb), size = 3.5, fontface = "bold"
  ) +
  coord_flip(ylim = c(0, y_bracket + 2), clip = "off") +
  scale_y_continuous(expand = c(0, 0)) +
  labs(
    title = "Net Benefit at 20% Threshold",
    subtitle = "Incident Type 2 Diabetes",
    y = "Net Benefit (per 1000 patients)",
    x = ""
  ) +
  theme(
    legend.position = "none",
    plot.title = element_text(size = 14, hjust = 0.5),
    plot.subtitle = element_text(size = 14, hjust = 0.5)
  )

# plot barplot 
p_t2d_bar
ggsave(
  file.path(fig_dir, "t2d_dca_barplot_20pct.png"),
  p_t2d_bar, width = 5, height = 2, dpi = 300
)


# Summarized Net Benefit

In [ ]:
# consolidate net benefit stats
# Net benefit is already per 1,000 patients on the 5-year curves.
# Delta NB is SOC+NMR minus SOC after rounding each to 2 decimals.
nb_thresholds <- c("5%" = 0.05, "10%" = 0.10, "20%" = 0.20)

net_benefit <- dplyr::bind_rows(lapply(primary_ids, function(id) {
  curves <- dca_curves[[id]]
  dplyr::bind_rows(lapply(names(nb_thresholds), function(lab) {
    nb_soc <- round(nb_at_threshold(curves$SOC, nb_thresholds[[lab]]), 2)
    nb_nmr <- round(nb_at_threshold(curves$`SOC+NMR`, nb_thresholds[[lab]]), 2)
    tibble::tibble(
      Endpoint = disease_specs[[id]]$title,
      Threshold = lab,
      `NB SOC` = nb_soc,
      `NB SOC+NMR` = nb_nmr,
      `Delta NB` = round(nb_nmr - nb_soc, 2)
    )
  }))
}))

net_benefit

# export
write.csv(
  net_benefit,
  here("outputs/tables", "DCA_NET_BENEFIT.csv"),
  row.names = FALSE
)

## Supplemental outcome: COPD

1-, 3-, and 5-year net benefit. COPD stays out of the primary grid.


In [ ]:
copd_horizons <- c("1 yr" = 1, "3 yr" = 3, "5 yr" = 5)
copd_x <- list("1 yr" = c(0.001, 0.30), "3 yr" = c(0.001, 0.50), "5 yr" = c(0.001, 0.50))
copd_y_lower <- c("1 yr" = -1, "3 yr" = -2, "5 yr" = -2)

p_copd <- lapply(names(copd_horizons), function(lab) {
  t_h <- copd_horizons[[lab]]
  curves <- if (t_h == t_star) {
    dca_curves$copd
  } else {
    lapply(oof_store$copd, function(obj) {
      compute_dca_df_from_oof_risk(
        obj,
        t_star = t_h,
        risk_col = sprintf("pred_risk_%dyr", t_h),
        threshold_range = threshold_range
      )
    })
  }
  p <- plot_dca_compare(
    curves,
    title = paste("COPD Net Benefit,", lab),
    x_limits = copd_x[[lab]],
    palette = model_palette,
    t_star = t_h
  )
  ymax <- p$coordinates$limits$y[2]
  p + coord_cartesian(ylim = c(unname(copd_y_lower[[lab]]), ymax))
})

options(repr.plot.height = 3.75, repr.plot.width = 10)
p_copd_row <- wrap_plots(p_copd, nrow = 1, guides = "collect") &
  theme(legend.position = "bottom")

p_copd_row
ggsave(
  file.path(fig_dir, "COPD_DCA.png"),
  p_copd_row, width = 10, height = 3.75, dpi = 300
)
